# GTO definition and integrals
We start this section assuming that all the recurrence schemes and special functions are well defined. Therefore we will apply them to compute GTO integrals.

We have to start by defining a GTO object. The input for the constructor must be:
- R (center)
- Alpha (ecponent)
- Total angular momentum (l)

However, the final GTO struct must possess not that, but the projecions, the normalization constants of each projection, and the charge. We start by defining the GTO dataclass:

In [1]:
from dataclasses import dataclass
import numpy as np
from numpy.typing import NDArray

@dataclass
class GTO:
    """
    Gaussian GTO basis function.

    Attributes
    ----------
    R : NDArray[np.float64]
        Center coordinates (3D vector)
    exp : float
        Gaussian exponent
    angular_momentum : int
        Angular momentum quantum number
    norm : float
        Normalization constant
    """

    R: NDArray[np.float64]
    exp: float
    total_L: int
    l_projections: NDArray[np.int32]  # of dimensions (n_projections, 3)
    normalization_constants: NDArray[np.float64]
    charge: float = 1.0

And we can define a `create_GTO` function: 


In [2]:
def create_GTO(R: NDArray[np.float64], exp: float, total_L: int) -> GTO:
    """
    Factory function to create a GTO object with computed angular momentum projections
    and normalization constants.

    Parameters

    ----------
    R : NDArray[np.float64]
        Center coordinates (3D vector)
    exp : float
        Gaussian exponent
    total_L : int
        Total angular momentum quantum number

    Returns
    -------
    GTO
        GTO object
    """
    # Generate all angular momentum projections for total_L
    l_projections = []
    for l_x in range(total_L + 1):
        for l_y in range(total_L - l_x + 1):
            l_z = total_L - l_x - l_y
            l_projections.append([l_x, l_y, l_z])
    l_projections = np.array(l_projections, dtype=np.int32)

    # Compute normalization constants for each projection
    normalization_constants = np.zeros(len(l_projections)) + 1

    prim = GTO(
        R=R,
        exp=exp,
        total_L=total_L,
        l_projections=l_projections,
        normalization_constants=normalization_constants,
    )

    return prim

We can create a GTO now of lets say $l=1$:


In [3]:
basis_1 = create_GTO(np.array([0,0,0]), 0.5, 3)

print(repr(basis_1))

GTO(R=array([0, 0, 0]), exp=0.5, total_L=3, l_projections=array([[0, 0, 3],
       [0, 1, 2],
       [0, 2, 1],
       [0, 3, 0],
       [1, 0, 2],
       [1, 1, 1],
       [1, 2, 0],
       [2, 0, 1],
       [2, 1, 0],
       [3, 0, 0]], dtype=int32), normalization_constants=array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]), charge=1.0)


In [4]:
print(basis_1.normalization_constants)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


However, we need to normalize the projections of the GTO. For this we can use the overlap function defined as:

In [5]:
from py_mods.src.integrals.internal.ST_utils import _S_1D_legacy as S_1D


def S_3D(
    basis_1: GTO,
    projection_1,
    N_A: float,
    basis_2: GTO,
    projection_2,
    N_B: float,
) -> float:
    """
    Calculate the product ofthe three overlap integral components.

    Parameters
    ------
    basis_1 : GTO
        First GTO; must provide attributes:
          - R : array-like of length 3, center coordinates (R_x, R_y, R_z)
          - exp : float, Gaussian exponent (alpha)
          - angular_momentum : int, total angular momentum (l)
    projection_1 : numpy.ndarray
        Length-3 integer array with projection quantum numbers for basis_1 (m_x, m_y, m_z)
    basis_2 : GTO
        Second GTO; same requirements as basis_1
    projection_2 : numpy.ndarray
        Length-3 integer array with projection quantum numbers for basis_2

    Returns
    ------
        float: The product of the three overlap components (S_ab, S_cd, S_ef).
    """
    S_ab, S_cd, S_ef = S_3D_components(basis_1, projection_1, basis_2, projection_2)

    return S_ab * S_cd * S_ef * N_A * N_B


# --- Overlap 3D with GTOs ---
def S_3D_components(
    basis_1: GTO,
    projection_1: np.ndarray,
    basis_2: GTO,
    projection_2: np.ndarray,
) -> np.ndarray:
    """
    Compute the three Cartesian components of the 3D overlap between two GTO functions.

    To ensure orthogonality if the scalar product is 0 (they dont share a
    component in the same projection) and the individual l is nonzero, the
    function returns [0,0,0].

    Parameters
    ----------
    basis_1 : GTO
        First GTO; must provide attributes:
          - R : array-like of length 3, center coordinates (R_x, R_y, R_z)
          - exp : float, Gaussian exponent (alpha)
          - angular_momentum : int, total angular momentum (l)
    projection_1 : numpy.ndarray
        Length-3 integer array with projection quantum numbers for basis_1 (m_x, m_y, m_z)
    basis_2 : GTO
        Second GTO; same requirements as basis_1
    projection_2 : numpy.ndarray
        Length-3 integer array with projection quantum numbers for basis_2

    Returns
    -------
    overlap_components : numpy.ndarray of shape (3,)
        1D numpy array of shape (3,) with the overlap components [S_x, S_y, S_z].
        If the projection vectors are orthogonal and both l1 and l2 are nonzero,
        returns numpy.array([0, 0, 0]).

    Notes
    -------
        - We will do it this way for now, since we have to test for d functions.
        It is true that the calculation of the three 1d overlaps might be redundant,
        but it must be checked out.
    """
    # If there is overlap calculate it
    R_a = basis_1.R
    R_b = basis_2.R

    alpha = basis_1.exp
    beta = basis_2.exp

    overlap_components = np.zeros(3)

    for comp, _ in enumerate(overlap_components):
        overlap_components[comp] = S_1D(
            R_a[comp], R_b[comp], alpha, beta, projection_1[comp], projection_2[comp]
        )

    return overlap_components

Lets try to compute the overlap between two projections of the same GTO:

In [6]:
print(
    S_3D(
        basis_1,
        basis_1.l_projections[0],
        basis_1.normalization_constants[0],
        basis_1,
        basis_1.l_projections[0],
        basis_1.normalization_constants[0],
    )
)

10.44061499405945


Then the normalization constant of tis projection is:

In [7]:
n_const_1 = 1 / np.sqrt(
    S_3D(
        basis_1,
        basis_1.l_projections[0],
        basis_1.normalization_constants[0],
        basis_1,
        basis_1.l_projections[0],
        basis_1.normalization_constants[0],
    )
)
n_const_1

np.float64(0.30948311499459147)

Now using this normalization constant:

In [8]:
print(
    S_3D(
        basis_1,
        basis_1.l_projections[0],
        n_const_1,
        basis_1,
        basis_1.l_projections[0],
        n_const_1,
    )
)

1.0000000000000002


Therefore we can define the `normalize_GTO` function as:

In [9]:
def normalize_GTO(Prim: GTO) -> None:
    """
    Normalize the GTO basis function in place by updating its normalization constants.

    Parameters
    ----------
    Prim : GTO
        The GTO basis function to be normalized. Its normalization_constants attribute
        will be updated in place.
    """
    for i, projection in enumerate(Prim.l_projections):
        N = 1 / np.sqrt(
            S_3D(
                Prim,
                projection,
                Prim.normalization_constants[i],
                Prim,
                projection,
                Prim.normalization_constants[i],
            )
        )
        Prim.normalization_constants[i] = N

So lets see if it works:

In [10]:
normalize_GTO(basis_1)
print(repr(basis_1))

GTO(R=array([0, 0, 0]), exp=0.5, total_L=3, l_projections=array([[0, 0, 3],
       [0, 1, 2],
       [0, 2, 1],
       [0, 3, 0],
       [1, 0, 2],
       [1, 1, 1],
       [1, 2, 0],
       [2, 0, 1],
       [2, 1, 0],
       [3, 0, 0]], dtype=int32), normalization_constants=array([0.30948311, 0.69202528, 0.69202528, 0.30948311, 0.69202528,
       1.19862295, 0.69202528, 0.69202528, 0.69202528, 0.30948311]), charge=1.0)


So the overlap between functions now is:

In [11]:
for i in range(len(basis_1.l_projections)):
    print(
        S_3D(
            basis_1,
            basis_1.l_projections[i],
            basis_1.normalization_constants[i],
            basis_1,
            basis_1.l_projections[i],
            basis_1.normalization_constants[i],
        )
    )

1.0000000000000002
1.0
0.9999999999999999
1.0000000000000002
1.0
0.9999999999999998
0.9999999999999999
0.9999999999999999
0.9999999999999999
1.0000000000000002


We create a generalized normalized GTO constructor:

In [12]:
def create_normalized_GTO(
    R: NDArray[np.float64], exp: float, total_L: int, charge: float =1 
) -> GTO:
    """
    Factory function to create a GTO object with computed angular momentum projections
    and normalization constants.

    Parameters

    ----------
    R : NDArray[np.float64]
        Center coordinates (3D vector)
    exp : float
        Gaussian exponent
    total_L : int
        Total angular momentum quantum number

    Returns
    -------
    GTO
        GTO object
    """
    # Generate all angular momentum projections for total_L
    l_projections = []
    for l_x in range(total_L + 1):
        for l_y in range(total_L - l_x + 1):
            l_z = total_L - l_x - l_y
            l_projections.append([l_x, l_y, l_z])
    l_projections = np.array(l_projections, dtype=np.int32)

    # Compute normalization constants for each projection
    normalization_constants = np.zeros(len(l_projections)) + 1

    prim = GTO(
        R=R,
        exp=exp,
        total_L=total_L,
        l_projections=l_projections,
        normalization_constants=normalization_constants,
        charge=charge
    )

    normalize_GTO(prim)

    return prim

## Overlap integrals
In order to normalize the GTO projections we had to compute the overlap integral between two Gaussian functions. Therefore, we can now try to see how to compute the overlap integral between two projections of two different GTOs. For that we will use four different GTOs located in different points: 
- GTO 1 at (0,0,0) with $l=0$
- GTO 2 at (0,0,0) with $l=1$
- GTO 3 at (1,0,0) with $l=0$
- GTO 4 at (1,0,0) with $l=2$

In [13]:
prim_1 = create_normalized_GTO(np.array([0.0, 0.0, 0.0]), 0.5, 0)
prim_2 = create_normalized_GTO(np.array([0.0, 0.0, 0.0]), 0.5, 1)
prim_3 = create_normalized_GTO(np.array([1.0, 0.0, 0.0]), 0.5, 0)
prim_4 = create_normalized_GTO(np.array([1.0, 0.0, 0.0]), 0.5, 1)

Firstm lets check for the same angular moment:

In [14]:
overlap_1 = S_3D(
    prim_1,
    prim_1.l_projections[0],
    prim_1.normalization_constants[0],
    prim_3,
    prim_3.l_projections[0],
    prim_3.normalization_constants[0],
)

print(f"s ([0,0,0]), s ([0,0,0]) = {overlap_1:.6f}")

s ([0,0,0]), s ([0,0,0]) = 0.778801


And for $l=1$

In [15]:
overlap_2 = S_3D(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[0],
    prim_4.normalization_constants[0],
)

print(f"p ([1,0,0]), p ([1,0,0]) = {overlap_2:.6f}")

p ([1,0,0]), p ([1,0,0]) = 0.778801


And with a different angular moment, or different angular moment projection, the overlap should be 0:

In [16]:
overlap_3 = S_3D(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[1],
    prim_4.normalization_constants[1],
)
print(f"p([1,0,0]), p ([0,1,0]) = {overlap_3:.2f}")


overlap_4 = S_3D(
    prim_1,
    prim_1.l_projections[0],
    prim_1.normalization_constants[0],
    prim_4,
    prim_4.l_projections[0],
    prim_4.normalization_constants[0],
)

print(f"s ([0,0,0]), p ([1,0,0]) = {overlap_4:.2f}")

p([1,0,0]), p ([0,1,0]) = 0.00
s ([0,0,0]), p ([1,0,0]) = 0.00


## Kinetic energy integrals
In the same way as overlap, the kinetic energy integral between two Gaussians is:

$$
T_{ab} = T_{ij} S_{kl} S_{mn} + S_{ij} T_{kl} S_{mn} + S_{ij} S_{kl} T_{mn}
$$

That implicitly contains the normalization constants of each projection. Then the formula becomes:

$$
T_{ab} = N_aN_b(T_{ij} S_{kl} S_{mn} + S_{ij} T_{kl} S_{mn} + S_{ij} S_{kl} T_{mn})
$$

We define the `T_3D` function as:


In [17]:
from py_mods.src.integrals.GTO import _T_3D_legacy as T_3D

The kinetic energy of two $l=0$ basis functions if they share projection components:

In [18]:
T_3D(
    prim_1,
    prim_1.l_projections[0],
    prim_1.normalization_constants[0],
    prim_3,
    prim_3.l_projections[0],
    prim_3.normalization_constants[0],
)

np.float64(0.486750489419628)

And two $l=1$ if they share projection:

In [19]:
T_3D(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[0],
    prim_4.normalization_constants[0],
)

np.float64(0.8761508809553304)

And if they dont:

In [20]:
T_3D(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[1],
    prim_4.normalization_constants[1],
)

np.float64(0.0)

## Potential energy integrals
Now the mission is to evaluate the potential energy integrals between two GTO Gaussians. The formula for the 3D potential energy integral is:


$$
h_{ab}^{NA} = -\sum_K Z_K V_{ab}^{000} = - \frac{2\pi}{p}\sum_{tuv}E_{tuv}^{ab} \sum_K Z_K R_{t,u,v}(p, \mathbf{R}_{PC_K})
$$

For unnormalized projections. Therefore, including the normalization constants we have:

$$
h_{ab}^{NA} = N_aN_b \left(-\sum_K Z_K V_{ab}^{000}\right) = N_aN_b \left(- \frac{2\pi}{p}\sum_{tuv}E_{tuv}^{ab} \sum_K Z_K R_{t,u,v}(p, \mathbf{R}_{PC_K})\right)
$$

We then define `V_3D` as:

In [21]:
from py_mods.src.integrals.GTO import _V_3D_legacy as V_3D

And we can calculate the integral with a $+1$ point charge located at (0,0,0) of two $s$ functions:

In [22]:
V_3D(
    prim_1,
    prim_1.l_projections[0],
    prim_1.normalization_constants[0],
    prim_3,
    prim_3.l_projections[0],
    prim_3.normalization_constants[0],
    1.0,
    np.array([0.0, 0.0, 0.0]) 
)

np.float64(-0.8107314248587427)

Between an $s$ and a $p$:

In [23]:
V_3D(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_1,
    prim_1.l_projections[0],
    prim_1.normalization_constants[0],
    1.0,
    np.array([0.0, 0.0, 0.0]) 
)

np.float64(-0.0)

And between $p$-functions of same and different projections of $l$:

In [24]:
V_3D(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[0],
    prim_4.normalization_constants[0],
    1.0,
    np.array([0.0, 0.0, 0.0]) 
)

np.float64(-0.5580616963901238)

In [25]:
V_3D(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[1],
    prim_4.normalization_constants[1],
    1.0,
    np.array([0.0, 0.0, 0.0]) 
)

np.float64(-0.0)

## GTO two electron repulsion integrals 

We can see the "simple" formula for the **unnormalized** two electron repulsion integral between four GTO Gaussians (Helgaker eq 9.9.33):

$$
( ab | cd ) = \frac{2\pi^{5/2}}{pq\sqrt{p+q}} \sum_{tuv} E_{tuv}^{ab} \sum_{t'u'v'} E_{t'u'v'}^{cd} R_{t+t',u+u',v+v'}(\alpha, \mathbf{R}_{PQ})
$$

where $\alpha = \frac{pq}{p+q}$ and $\mathbf{R}_{PQ} = \mathbf{P} - \mathbf{Q}$, with $\mathbf{P}$ and $\mathbf{Q}$ being the Gaussian product centers of the $ab$ and $cd$ pairs respectively, $E_{tuv}^{ab}$ and $E_{t'u'v'}^{cd}$ the Hermite coefficients for each pair, and $R_{t+t',u+u',v+v'}(\alpha, \mathbf{R}_{PQ})$ the Hermite Coulomb integral.

In [26]:
from py_mods.src.integrals.GTO import _g_abcd_legacy as g_abcd
from py_mods.src.integrals.GTO import _eri_legacy as eri

We can calculate the eri between four $s$ functions located at (0,0,0) and (1,0,0):

$$
(s_1s_1|s_2s_2) = N_{s_1}^2 N_{s_2}^2\frac{2\pi^{5/2}}{pq\sqrt{p+q}} R_{000}(\alpha, \mathbf{R}_{PQ})
$$

In [27]:
eri(
    prim_1,
    prim_1.l_projections[0],
    prim_1.normalization_constants[0],
    prim_1,
    prim_1.l_projections[0],
    prim_1.normalization_constants[0],
    prim_3,
    prim_3.l_projections[0],
    prim_3.normalization_constants[0],
    prim_3,
    prim_3.l_projections[0],
    prim_3.normalization_constants[0],
)

np.float64(0.6826894921370865)

And between two $p$ functions separated by 1 unit along the x axis with the same projections:

In [28]:
eri(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[0],
    prim_4.normalization_constants[0],
    prim_4,
    prim_4.l_projections[0],
    prim_4.normalization_constants[0],
)

np.float64(0.5681684592318701)

And with different projections:

In [29]:
eri(
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_2,
    prim_2.l_projections[0],
    prim_2.normalization_constants[0],
    prim_4,
    prim_4.l_projections[1],
    prim_4.normalization_constants[1],
    prim_4,
    prim_4.l_projections[1],
    prim_4.normalization_constants[1],
)

np.float64(0.5120171191028146)